# Pruebas comparativas — PostgreSQL con pgvector
Usa las mismas 20 frases de tickets bancarios, 5 preguntas, modelo, semilla (42), `N=20000` y parámetros HNSW (`M=16`, `ef_construction=100`, `ef_search=64`) que Qdrant y Chroma. Guarda el resultado en `resultados_pgvector.txt`.


In [1]:
import os
import statistics
import time
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")

import numpy as np
from sentence_transformers import SentenceTransformer

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REPETICIONES = 10
N_BENCH = 20000
SEMILLA = 42
M, EF_CONSTRUCTION, EF_SEARCH = 16, 100, 64

documentos = [
    "El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático",
    "Solicitud de aumento de línea de crédito para cuenta empresarial",
    "El cliente no reconoce un cargo de $150 en su estado de cuenta",
    "Consulta sobre requisitos para abrir una cuenta de ahorros infantil",
    "Reclamo por doble cobro en transferencia interbancaria",
    "Solicitud de copia de estado de cuenta de los últimos 6 meses",
    "El cajero automático retuvo la tarjeta del cliente",
    "Consulta sobre tasa de interés para préstamo hipotecario",
    "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito",
    "Solicitud para cambiar el número de teléfono asociado a la cuenta",
    "El cliente reporta una transacción que no ha realizado con su tarjeta de crédito",
    "No está funcionando las transferencias ACH a las 08:00 am",
    "El usuario de banca en línea fue bloqueado y no puede entrar",
    "La app móvil se encuentra fuera de servicio",
    "El cliente solicita un préstam por consolidación de deudas",
    "La empresa ABC quiere cancelar su cuenta de cheques y abrir una nueva",
    "Se han incrementado las solicitudes de tarjeta de crédto para clientes con historial crediticio limitado",
    "Consulta para cancelar una tarjeta de crédito que ha sido robada",
    "El usuario de la app no puede ver el saldo de la tarjeta de crédito",
    "Solicitud para préstamo de pequeña empresa para capital de trabajo",
]

categorias = [
    "tarjetas", "prestamos", "fraude", "cuentas", "transferencias",
    "cuentas", "tarjetas", "prestamos", "fraude", "cuentas",
    "fraude", "canales_digitales", "canales_digitales", "canales_digitales", "prestamos",
    "cuentas", "tarjetas", "tarjetas", "canales_digitales", "prestamos",
]

preguntas_prueba = [
    ("alguien uso mi tarjeta sin permiso", "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito"),
    ("me quedé sin poder entrar a la banca en línea", "El usuario de banca en línea fue bloqueado y no puede entrar"),
    ("quiero un crédito para mi negocio", "Solicitud para préstamo de pequeña empresa para capital de trabajo"),
    ("el cajero se tragó mi tarjeta", "El cajero automático retuvo la tarjeta del cliente"),
    ("necesito mis estados de cuenta anteriores", "Solicitud de copia de estado de cuenta de los últimos 6 meses"),
]

lineas = []
def log(msg=""):
    print(msg)
    lineas.append(str(msg))

log("Cargando modelo de embeddings...")
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedding_docs = modelo.encode(documentos).astype("float32")
DIM = embedding_docs.shape[1]
log(f"Forma de los embeddings: {embedding_docs.shape}")

SALIDA_TXT = RAIZ / "resultados_pgvector.txt"


Cargando modelo de embeddings...
Forma de los embeddings: (20, 384)


In [2]:
import psycopg
from pgvector.psycopg import register_vector

DSN = "host=localhost port=5432 dbname=vectordb user=postgres password=postgres"
conn = psycopg.connect(DSN, autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(conn)

version = conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0]
conn.execute(f"SET hnsw.ef_search = {EF_SEARCH}")
log(f"Base: pgvector ({version}) | HNSW M={M}, ef_construction={EF_CONSTRUCTION}, ef_search={EF_SEARCH}")

TABLA = "tickets_banco"
conn.execute(f"DROP TABLE IF EXISTS {TABLA}")
conn.execute(f"""
    CREATE TABLE {TABLA} (
        id integer PRIMARY KEY,
        texto text,
        categoria text,
        embedding vector({DIM})
    )
""")

t0 = time.perf_counter()
with conn.cursor() as cur:
    cur.executemany(
        f"INSERT INTO {TABLA} (id, texto, categoria, embedding) VALUES (%s, %s, %s, %s)",
        [(i, doc, cat, emb) for i, (doc, cat, emb) in enumerate(zip(documentos, categorias, embedding_docs))],
    )
t_carga = time.perf_counter() - t0
conn.execute(f"CREATE INDEX idx_{TABLA}_categoria ON {TABLA} (categoria)")
total_frases = conn.execute(f"SELECT count(*) FROM {TABLA}").fetchone()[0]
log(f"\n[1] Carga de {total_frases} frases: {t_carga:.3f} s")


Base: pgvector (0.8.7) | HNSW M=16, ef_construction=100, ef_search=64

[1] Carga de 20 frases: 0.050 s


In [3]:
def buscar(consulta_vec, k, categoria=None):
    where = "WHERE categoria = %s" if categoria else ""
    sql = f"""
        SELECT texto, 1 - (embedding <=> %s::vector) AS similitud
        FROM {TABLA}
        {where}
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """
    params = (consulta_vec, categoria, consulta_vec, k) if categoria else (consulta_vec, consulta_vec, k)
    filas = conn.execute(sql, params).fetchall()
    return [f[0] for f in filas], [float(f[1]) for f in filas]

vectores_q = [modelo.encode(p).astype("float32") for p, _ in preguntas_prueba]
buscar(vectores_q[0], 3)  # calentamiento

log("\n[2] Aciertos y tiempos (solo la consulta a pgvector)")
resumen = {}
for k in (3, 5, 10):
    aciertos, tiempos = 0, []
    for (pregunta, esperado), v in zip(preguntas_prueba, vectores_q):
        textos, _ = buscar(v, k)
        if esperado in textos:
            aciertos += 1
        for _ in range(REPETICIONES):
            t0 = time.perf_counter()
            buscar(v, k)
            tiempos.append((time.perf_counter() - t0) * 1000)
    resumen[k] = (aciertos, min(tiempos), max(tiempos), statistics.mean(tiempos))
    log(f"  K={k:<2} aciertos {aciertos}/{len(preguntas_prueba)} | "
        f"min {min(tiempos):.1f} | max {max(tiempos):.1f} | "
        f"prom {statistics.mean(tiempos):.1f} ms")

log("\n  Detalle (K=3):")
for (pregunta, esperado), v in zip(preguntas_prueba, vectores_q):
    textos, scores = buscar(v, 3)
    log(("  OK  " if esperado in textos else "  FALLO") + " " + pregunta)
    if esperado not in textos:
        log("     Esperado: " + esperado)
        for t, s in zip(textos, scores):
            log(f"     {s:.3f} | {t}")



[2] Aciertos y tiempos (solo la consulta a pgvector)
  K=3  aciertos 4/5 | min 0.6 | max 1.8 | prom 0.9 ms
  K=5  aciertos 4/5 | min 0.6 | max 2.1 | prom 0.9 ms
  K=10 aciertos 5/5 | min 0.5 | max 1.5 | prom 0.8 ms

  Detalle (K=3):
  FALLO alguien uso mi tarjeta sin permiso
     Esperado: Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
     0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
     0.602 | El cajero automático retuvo la tarjeta del cliente
     0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
  OK   me quedé sin poder entrar a la banca en línea
  OK   quiero un crédito para mi negocio
  OK   el cajero se tragó mi tarjeta
  OK   necesito mis estados de cuenta anteriores


In [4]:
log("\n[3] Filtro por metadatos")
v = modelo.encode("alguien uso mi tarjeta sin permiso").astype("float32")
log("  SIN filtro:")
for t, s in zip(*buscar(v, 3)):
    log(f"    {s:.3f} | {t}")
log("  CON filtro categoria=fraude:")
for t, s in zip(*buscar(v, 3, categoria="fraude")):
    log(f"    {s:.3f} | {t}")



[3] Filtro por metadatos
  SIN filtro:
    0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
    0.602 | El cajero automático retuvo la tarjeta del cliente
    0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
  CON filtro categoria=fraude:
    0.576 | El cliente reporta una transacción que no ha realizado con su tarjeta de crédito
    0.491 | Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
    0.411 | El cliente no reconoce un cargo de $150 en su estado de cuenta


In [5]:
BENCH = "bench"
conn.execute(f"DROP TABLE IF EXISTS {BENCH}")
conn.execute(f"CREATE TABLE {BENCH} (id integer PRIMARY KEY, embedding vector({DIM}))")

rng = np.random.default_rng(SEMILLA)
datos = rng.normal(size=(N_BENCH, DIM)).astype("float32")

t0 = time.perf_counter()
with conn.cursor() as cur:
    for i in range(0, N_BENCH, 1000):
        fin = min(i + 1000, N_BENCH)
        cur.executemany(
            f"INSERT INTO {BENCH} (id, embedding) VALUES (%s, %s)",
            [(j, datos[j]) for j in range(i, fin)],
        )
t_bench = time.perf_counter() - t0
conn.execute(f"ANALYZE {BENCH}")

t0 = time.perf_counter()
conn.execute(f"CREATE INDEX idx_{BENCH}_hnsw ON {BENCH} USING hnsw (embedding vector_cosine_ops) WITH (m = {M}, ef_construction = {EF_CONSTRUCTION})")
conn.execute(f"ANALYZE {BENCH}")
t_indice = time.perf_counter() - t0
log(f"\n[4] Carga de {N_BENCH} vectores: {t_bench:.2f} s | Índice HNSW: {t_indice:.2f} s (total {t_bench + t_indice:.2f} s)")

SQL_BENCH = f"SELECT id FROM {BENCH} ORDER BY embedding <=> %s::vector LIMIT 10"
consulta = rng.normal(size=DIM).astype("float32")
tiempos = []
for _ in range(100):
    t0 = time.perf_counter()
    conn.execute(SQL_BENCH, (consulta,)).fetchall()
    tiempos.append((time.perf_counter() - t0) * 1000)
log(f"    Latencia con {N_BENCH} vectores: promedio {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")



[4] Carga de 20000 vectores: 2.41 s | Índice HNSW: 12.47 s (total 14.88 s)
    Latencia con 20000 vectores: promedio 2.2 ms | p95 3.1 ms


In [6]:
norm = datos / np.linalg.norm(datos, axis=1, keepdims=True)
aciertos, total = 0, 0
for _ in range(50):
    q = rng.normal(size=DIM).astype("float32")
    aprox = {int(r[0]) for r in conn.execute(SQL_BENCH, (q,)).fetchall()}
    sims = norm @ (q / np.linalg.norm(q))
    exacta = set(np.argsort(-sims)[:10].tolist())
    aciertos += len(aprox & exacta)
    total += 10
recall = aciertos / total
log(f"[5] Recall@10 (HNSW vs exacta): {recall:.2%}")

log("\n================ RESUMEN PGVECTOR ================")
log(f"Carga 20 frases:        {t_carga:.3f} s")
for k, (a, mn, mx, pr) in resumen.items():
    log(f"K={k:<2} aciertos {a}/5 | tiempo min/max/prom: {mn:.1f} / {mx:.1f} / {pr:.1f} ms")
log(f"Carga {N_BENCH} vectores: {t_bench:.2f} s (+ índice {t_indice:.2f} s)")
log(f"Latencia {N_BENCH} (prom): {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")
log(f"Recall@10:              {recall:.2%}")

SALIDA_TXT.write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(f"\nGuardado en {SALIDA_TXT}")
conn.close()


[5] Recall@10 (HNSW vs exacta): 26.60%

================ RESUMEN PGVECTOR ================
Carga 20 frases:        0.050 s
K=3  aciertos 4/5 | tiempo min/max/prom: 0.6 / 1.8 / 0.9 ms
K=5  aciertos 4/5 | tiempo min/max/prom: 0.6 / 2.1 / 0.9 ms
K=10 aciertos 5/5 | tiempo min/max/prom: 0.5 / 1.5 / 0.8 ms
Carga 20000 vectores: 2.41 s (+ índice 12.47 s)
Latencia 20000 (prom): 2.2 ms | p95 3.1 ms
Recall@10:              26.60%

Guardado en C:\Users\USER\Proyectos\proyecto_BDII\resultados_pgvector.txt
